## introducccion a langchain

In [1]:
import langchain_core
print(langchain_core.__version__)

1.4.0


### Iniciaizamos el modelo de lenguaje

In [2]:
import os
with open("api_key.txt") as archivo:
  apikey = archivo.read()
os.environ["OPENAI_API_KEY"] = apikey

In [3]:
modelo_base = "gpt-4o-mini"

- Forma 1: Forma generica

In [4]:
from langchain.chat_models import init_chat_model

model = init_chat_model(modelo_base, model_provider="openai")


In [5]:
respuesta = model.invoke("Cual es la capital de Peru")
respuesta


AIMessage(content='La capital de Perú es Lima.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 14, 'total_tokens': 21, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_2767f2907f', 'id': 'chatcmpl-DmJo3jGjzL2Q80dwnvyDMwOVSlWV2', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e8899-66a0-7cd1-a935-37a039d151db-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 7, 'total_tokens': 21, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [6]:
print(f"Texto: {respuesta.content}")
print(f"Tokens usados: {respuesta.usage_metadata['total_tokens']}")
print(f"Modelo real usado: {respuesta.response_metadata['model_name']}")

Texto: La capital de Perú es Lima.
Tokens usados: 21
Modelo real usado: gpt-4o-mini-2024-07-18


- Forma 1: Forma generica (Ollama)

In [7]:
import os

# Prevenir bloqueos de conexión local en Windows
os.environ["HTTP_PROXY"] = ""
os.environ["HTTPS_PROXY"] = ""
os.environ["NO_PROXY"] = "127.0.0.1,localhost,::1"

from langchain.chat_models import init_chat_model

# Configuración del modelo base local
modelo_base2 = "gemma3:4b"

# Inicialización unificada mediante la fábrica de LangChain
model2 = init_chat_model(
    modelo_base2,
    model_provider="ollama",
    base_url="http://127.0.0.1:11434",  # Forzar IPv4 local
    temperature=0.1
)

# Ejecución de la consulta
respuesta2 = model2.invoke("Cual es la capital de Peru")

# Impresión de metadatos del objeto de respuesta
print(f"Texto: {respuesta2.content}")
print(f"Tokens usados: {respuesta2.usage_metadata['total_tokens']}")
print(f"Modelo real usado: {respuesta2.response_metadata['model_name']}")

Texto: La capital de Perú es **Lima**.

Tokens usados: 26
Modelo real usado: gemma3:4b


- Forma 2

In [7]:
from langchain_openai import ChatOpenAI
modelo_base = "gpt-4o-mini"
model2 = ChatOpenAI(model=modelo_base)


In [8]:
respuesta = model2.invoke("Cual es la capital de Peru")
respuesta


AIMessage(content='La capital de Perú es Lima.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 14, 'total_tokens': 21, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_2767f2907f', 'id': 'chatcmpl-DmJsEzVlrGfGFQNpH8oKoC0GxAkFG', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e889d-5e0d-77a3-b1d2-95081ce0d8a0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 7, 'total_tokens': 21, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [9]:
print(f"Texto: {respuesta.content}")
print(f"Tokens usados: {respuesta.usage_metadata['total_tokens']}")
print(f"Modelo real usado: {respuesta.response_metadata['model_name']}")

Texto: La capital de Perú es Lima.
Tokens usados: 21
Modelo real usado: gpt-4o-mini-2024-07-18


### Flujo de cadena simple(Ejemplo Simple de Runnables)

In [10]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate

1. Crear el Prompt
- Primero defines cómo quieres que sea la instrucción para el modelo.

In [11]:
prompt = ChatPromptTemplate.from_template("¿Cuál es la capital de : {variable_jorge}?")

2. Elegir el LLM
- Después eliges el modelo que procesará ese prompt.

In [12]:
llm = init_chat_model("gpt-4o-mini", model_provider="openai")

3. Conectar el Prompt y el LLM
- Ahora conectamos el prompt y el modelo usando el operador |.
- | (Esto es un Runnable lo cual veremos mas adelante)

In [15]:
chain_jorge = prompt | llm

4. Invocar la Cadena
- Finalmente, envías un input para obtener la respuesta

In [16]:
respuesta_modelo = chain_jorge.invoke({"variable_jorge": "Brasil"})

In [17]:
print(respuesta_modelo.content)

La capital de Brasil es Brasilia. Fue inaugurada como capital en 1960, diseñada por el arquitecto Oscar Niemeyer y el urbanista Lúcio Costa.


In [18]:
print(respuesta_modelo.usage_metadata['total_tokens'])
print(respuesta_modelo.response_metadata["model_name"])

51
gpt-4o-mini-2024-07-18


---- ahora con ollama ----

In [18]:
# Definición del prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un bot de geografía. Responde en una palabra."),
    ("human", "¿Cuál es la capital de: {country}?")
])
#diferencia entre from_template y from_messages:
# from_template se utilizaba en unos inicios (no te permite el system y human).
# from_messages es mas actualizado y te permite de tener roles.



# Inicialización unificada con el proveedor Ollama
llm = init_chat_model(
    "gemma3:4b", 
    model_provider="ollama",
    base_url="http://127.0.0.1:11434"  # Forzar IPv4 local
)

# Construcción de la cadena
chain = prompt | llm

# Invocación
response = chain.invoke({"country": "Brasil"})

# Impresión de resultados y metadatos
print(response.content)
print(response.usage_metadata['total_tokens'])
print(response.response_metadata["model_name"])

Brasilia.
40
gemma3:4b


---- aplicacion ----

In [19]:
import pandas as pd

data = {  "pais": ["Brasil", "Francia", "Japón", "Canadá", "Australia"]}
df = pd.DataFrame(data)
df

,pais
0,Brasil
1,Francia
2,Japón
3,Canadá
4,Australia


In [20]:

# 1. Inicialización del modelo y cadena
prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un bot de geografía. Responde estrictamente con una sola palabra (el nombre de la capital)."),
    ("human", "¿Cuál es la capital de: {country}?")
])

llm = init_chat_model(
    "gpt-4o-mini", 
    model_provider="openai"
)

chain = prompt | llm
#La chain va fuera de la función para un uso optimo de los recursos (RAM)


# 2. Definición de la función de consulta
def obtener_capital(pais: str) -> str:
    try:
        response = chain.invoke({"country": pais})
        return response.content.strip()
    except Exception as e:
        return f"Error: {e}"


# 4. Aplicación vectorizada de la función para crear la nueva columna
df["capital"] = df["pais"].apply(obtener_capital)

# Mostrar el resultado final
df

,pais,capital
0,Brasil,Brasilia
1,Francia,París
2,Japón,Tokio
3,Canadá,Ottawa
4,Australia,Canberra


## Ejemplos usando RunnableSequence y el conector "|"

### Ejemplo de secuencia: RunnableSequence

In [21]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableSequence

In [22]:
pelicula_prompt = ChatPromptTemplate.from_template("Devuelve solo el nombre de una película del género: {genero_pelicula}")

In [23]:
llm = init_chat_model("gpt-4o-mini", model_provider="openai")

In [24]:
explain_prompt = ChatPromptTemplate.from_template("Genera la descripción de la película en 30 palabras: {pelicula}")

In [25]:
def extract_content(message):
        return {"pelicula": message.content}

In [27]:
chain = RunnableSequence(
                              pelicula_prompt
                              | llm
                              | RunnableLambda(extract_content)
                              | explain_prompt
                              | llm
                            )

In [28]:
result = chain.invoke({"genero_pelicula": "Terror de los años 80"})

In [29]:
print(result.content)

"El resplandor" sigue a Jack Torrance, un escritor que se transforma en un peligroso lunático mientras cuida un hotel aislado. Su hijo, Danny, posee habilidades psíquicas que revelan oscuros secretos.


--- con ollama ---

In [30]:
import os

# 1. Configuración de red para evitar interferencias de proxies locales en Windows
os.environ["HTTP_PROXY"] = ""
os.environ["HTTPS_PROXY"] = ""
os.environ["NO_PROXY"] = "127.0.0.1,localhost,::1"

from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableSequence
from langchain_core.output_parsers import StrOutputParser

# 2. Definición de Prompts (Se mantiene la compatibilidad con tu lógica secuencial)
pelicula_prompt = ChatPromptTemplate.from_template(
    "Devuelve solo el nombre de una película del género: {genero_pelicula}"
)
explain_prompt = ChatPromptTemplate.from_template(
    "Genera la descripción de la película en 30 palabras: {pelicula}"
)

# 3. Inicialización unificada con el proveedor Ollama
llm = init_chat_model(
    "gemma3:4b", 
    model_provider="ollama",
    base_url="http://127.0.0.1:11434",  # Forzar IPv4 local
    temperature=0.7                    # Añadimos un poco de variabilidad creativa para géneros
)

# 4. Función de extracción de contenido (conversión a diccionario para el siguiente prompt)
def extract_content(message):
    return {"pelicula": message.content}

# 5. Construcción de la cadena secuencial funcional
# Nota: Añadimos StrOutputParser al final para retornar directamente el texto plano (str)
chain = RunnableSequence(
    pelicula_prompt
    | llm
    | RunnableLambda(extract_content)
    | explain_prompt
    | llm
    | StrOutputParser()
)

# 6. Invocación y ejecución
result = chain.invoke({"genero_pelicula": "Terror de los años 80"})
print(result)

En una ciudad plagada de pesadillas, un grupo de adolescentes es atormentado por Freddy Krueger, un asesino que solo puede ser derrotado en el mundo de los sueños.


-- aplicacion--


In [31]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableSequence
from langchain_core.output_parsers import StrOutputParser

# 2. Definición de Prompts (Ámbito global para eficiencia computacional)
pelicula_prompt = ChatPromptTemplate.from_template(
    "Devuelve solo el nombre de una película del género: {genero_pelicula}"
)
explain_prompt = ChatPromptTemplate.from_template(
    "Genera la descripción de la película en 30 palabras: {pelicula}"
)

# 3. Inicialización del modelo con el proveedor Ollama
llm = init_chat_model(
    "gemma3:4b", 
    model_provider="ollama",
    base_url="http://127.0.0.1:11434",
    temperature=0.7
)

# 4. Construcción de cadenas intermedias para capturar ambas salidas
chain_nombre = pelicula_prompt | llm | StrOutputParser()


def formatear_explicacion(input_data):
    return {"pelicula": input_data.strip()}

chain_descripcion = RunnableLambda(formatear_explicacion) | explain_prompt | llm | StrOutputParser()

chain_completa = chain_nombre | RunnableLambda(lambda nombre: {"nombre": nombre, "descripcion": chain_descripcion.invoke(nombre)})


# 5. Función modular de procesamiento para Pandas
def procesar_genero(genero: str):
    """
    Ejecuta el pipeline secuencial de LangChain para un género dado.
    Retorna una estructura pd.Series con el nombre y la descripción para mapeo directo.
    """
    try:
        # Invocación de la cadena unificada
        resultado = chain_completa.invoke({"genero_pelicula": genero})
        
        # Retornamos una serie indicando los nombres de los campos de salida
        return pd.Series([resultado["nombre"].strip(), resultado["descripcion"].strip()])
    except Exception as e:
        return pd.Series([f"Error: {e}", f"Error: {e}"])



In [32]:

# 6. Creación del Dataset de prueba
data = {
    "genero_pelicula": [
        "Terror de los años 80", 
        "Ciencia ficción Cyberpunk", 
        "Comedia romántica de los 90",
        "Documental de naturaleza"
    ]
}
df = pd.DataFrame(data)
df


,genero_pelicula
0,Terror de los años 80
1,Ciencia ficción Cyberpunk
2,Comedia romántica de los 90
3,Documental de naturaleza


In [33]:
# 7. Aplicación vectorizada por filas con asignación múltiple de columnas

df[["nombre_pelicula", "descripcion_pelicula"]] = df["genero_pelicula"].apply(procesar_genero)

df

,genero_pelicula,nombre_pelicula,descripcion_pelicula
0,Terror de los años 80,Pesadilla en la calle Elm,"Claro, aquí tienes una descripción de la pelíc..."
1,Ciencia ficción Cyberpunk,Blade Runner.,"En un futuro distópico, un Blade Runner, Rick ..."
2,Comedia romántica de los 90,Cuando Harry encontró a Sally.,"Claro, aquí tienes una descripción de la pelíc..."
3,Documental de naturaleza,Planeta Tierra,"En un futuro distópico, la humanidad se enfren..."
